# Bioinformatics Data Formats: A Comprehensive Guide

## Tier 3 – Applied Bioinformatics

Modern bioinformatics pipelines exchange data through a rich ecosystem of
plain-text and binary file formats. This notebook surveys **every major format**
you will encounter in NGS, structural biology, and phylogenetics work, with
hands-on Python parsing examples for each one.

---

### Contents

1. [FASTA – Sequences](#1.-FASTA)
2. [FASTQ – Sequencing Reads](#2.-FASTQ)
3. [SAM / BAM / CRAM – Alignments](#3.-SAM-/-BAM-/-CRAM)
4. [VCF / BCF – Variants](#4.-VCF-/-BCF)
5. [BED – Genomic Intervals](#5.-BED)
6. [GFF / GTF – Gene Annotations](#6.-GFF-/-GTF)
7. [WIG / BedGraph / BigWig – Coverage Tracks](#7.-WIG-/-BedGraph-/-BigWig)
8. [PDB – Protein Structure](#8.-PDB)
9. [mmCIF / PDBx – Modern Structure Format](#9.-mmCIF-/-PDBx)
10. [FAST5 / POD5 – Oxford Nanopore Raw Signal](#10.-FAST5-/-POD5)
11. [Newick / Nexus / NHX – Phylogenetic Trees](#11.-Newick-/-Nexus-/-NHX)
12. [Format Quick-Reference Table](#12.-Quick-Reference)


---

[< Previous: NGS Fundamentals: From Sequencing to Alignment](01_ngs_fundamentals.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Variant Calling and SNP Analysis >](../02_Variant_Calling_and_SNP_Analysis/01_variant_calling_and_snp_analysis.ipynb)

---

---

## Imports


In [ ]:
# Standard-library and lightweight helpers used throughout this notebook
import io
import re
import gzip
import struct
import textwrap
from pathlib import Path
from collections import defaultdict

# BioPython covers most format I/O we need
try:
    from Bio import SeqIO, AlignIO
    from Bio.SeqRecord import SeqRecord
    from Bio.Seq import Seq
    BIOPYTHON = True
except ImportError:
    BIOPYTHON = False
    print("BioPython not installed – pure-Python fallbacks will be used.")

print("Ready.")


---

## 1. FASTA

### 1.1 Format specification

FASTA is the simplest sequence format and the *lingua franca* of bioinformatics.

```
>sequence_id [optional description]
AGCTAGCTAGCTAGCTAGCT
AGCTAGCTAGCTAGCTAGCT   ← wrap at 60–80 chars (conventional)
```

Key rules
- The `>` character **must** be the first character on the header line.
- Sequence lines may span multiple lines; there is no canonical line length.
- Blank lines between records are **allowed but discouraged**.
- Sequence characters are case-insensitive; lowercase is used for soft-masked
  (repeat-masked) regions.

### 1.2 Common variants

| Variant | Extension | Notes |
|---------|-----------|-------|
| Nucleotide sequences | `.fa`, `.fna`, `.fasta` | DNA/RNA |
| Protein sequences | `.faa`, `.fasta` | Amino-acid one-letter codes |
| Multi-FASTA | `.fa`, `.fasta` | Multiple records in one file |
| FASTA quality (QUAL) | `.qual` | Phred scores in FASTA-style (legacy) |
| Indexed FASTA | `.fa` + `.fai` | `samtools faidx` index for random access |


In [ ]:
# ── Pure-Python FASTA parser ──────────────────────────────────────────────
def parse_fasta(text_or_path):
    """Yield (header, sequence) tuples from a FASTA string or file path."""
    if isinstance(text_or_path, (str, Path)) and Path(text_or_path).exists():
        opener = gzip.open if str(text_or_path).endswith('.gz') else open
        with opener(text_or_path, 'rt') as fh:
            yield from _fasta_iter(fh)
    else:
        yield from _fasta_iter(io.StringIO(str(text_or_path)))

def _fasta_iter(fh):
    header, seqs = None, []
    for line in fh:
        line = line.rstrip('\n')
        if line.startswith('>'):
            if header is not None:
                yield header, ''.join(seqs)
            header, seqs = line[1:], []
        elif header is not None:
            seqs.append(line)
    if header is not None:
        yield header, ''.join(seqs)

# ── Example FASTA data ────────────────────────────────────────────────────
sample_fasta = """\
>sp|P69905|HBA_HUMAN Hemoglobin subunit alpha OS=Homo sapiens
MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHG
KKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTP
AVHASLDKFLASVSTVLTSKYR
>sp|P68871|HBB_HUMAN Hemoglobin subunit beta OS=Homo sapiens
MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPK
VKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFG
KEFTPPVQAAYQKVVAGVANALAHKYH
"""

for header, seq in parse_fasta(sample_fasta):
    seq_id = header.split()[0]
    print(f"ID   : {seq_id}")
    print(f"Desc : {' '.join(header.split()[1:])}")
    print(f"Len  : {len(seq)} aa")
    print()


In [ ]:
# ── Writing FASTA ─────────────────────────────────────────────────────────
def write_fasta(records, wrap=60):
    """Return a FASTA-formatted string from an iterable of (header, seq) pairs."""
    lines = []
    for header, seq in records:
        lines.append(f'>{header}')
        for i in range(0, len(seq), wrap):
            lines.append(seq[i:i+wrap])
    return '\n'.join(lines) + '\n'

output = write_fasta([
    ("my_gene|123 A synthetic demo sequence", "ATGCGTACGATCGATCGATCGATCGATCGATCGATCGATCGATCG"),
    ("another_gene|456 Second record",         "ATCGATCGATCGATCGATCG"),
])
print(output)


In [ ]:
# ── FASTA index (.fai) ────────────────────────────────────────────────────
# samtools faidx creates a 5-column tab-separated index:
# NAME  LENGTH  OFFSET  LINEBASES  LINEWIDTH

fai_example = """\
chr1\t248956422\t52\t60\t61
chr2\t242193529\t253404903\t60\t61
chrX\t156040895\t2699520056\t60\t61
"""

print("FAI column meanings:")
cols = ["NAME", "LENGTH", "OFFSET (bytes)", "BASES PER LINE", "BYTES PER LINE"]
for col, val in zip(cols, fai_example.strip().split('\n')[0].split('\t')):
    print(f"  {col:<20} = {val}")

# Random-access using the index
def faidx_fetch(fasta_path, fai_path, chrom, start, end):
    """
    Fetch a substring from an indexed FASTA without loading the whole file.
    Coordinates are 0-based half-open [start, end).
    """
    # Load index
    index = {}
    with open(fai_path) as fh:
        for line in fh:
            name, length, offset, bases_per_line, bytes_per_line = line.split()
            index[name] = (int(length), int(offset),
                           int(bases_per_line), int(bytes_per_line))
    length, offset, bases_per_line, bytes_per_line = index[chrom]
    # Byte position of first base of the query region
    n_full_lines, remainder = divmod(start, bases_per_line)
    byte_start = offset + n_full_lines * bytes_per_line + remainder
    # Upper bound (over-estimate; we'll trim)
    bases_needed = end - start
    bytes_needed = (bases_needed // bases_per_line + 2) * bytes_per_line
    with open(fasta_path, 'rb') as fh:
        fh.seek(byte_start)
        raw = fh.read(bytes_needed).decode()
    seq = re.sub(r'\s', '', raw)[:bases_needed]
    return seq

print("\nfaidx_fetch is defined – call it on a real .fa/.fai pair to fetch regions.")


---

## 2. FASTQ

### 2.1 Format specification

FASTQ extends FASTA by embedding per-base quality scores.

```
@SEQ_ID [optional description]    ← header line (@ not >)
AGCTAGCTAGCTAGCT                  ← raw sequence (may NOT wrap)
+                                 ← separator (optionally repeats the header)
IIIIIIIIIIIIIIII                  ← quality string, same length as sequence
```

Each ASCII character in the quality string encodes a **Phred score**:

```
Phred(Q) = -10 × log₁₀(P_error)
ASCII offset 33 (Sanger/Illumina ≥1.8): quality_char = chr(Q + 33)
```

### 2.2 Quality encoding history

| Encoding | Offset | Range | Used by |
|----------|--------|-------|---------|
| Sanger / Illumina ≥1.8 | 33 | Q0–Q40 | Current standard |
| Illumina 1.3–1.7 | 64 | Q0–40 | Legacy |
| Solexa | 64 | Q-5 to Q40 | Very old data |

> 💡 **Tip**: `file.fastq.gz` gzip-compressed FASTQ is the de-facto standard
> for storing raw reads; always gzip before storing.


In [ ]:
# ── FASTQ parser and quality stats ───────────────────────────────────────
import math

def parse_fastq(source, max_reads=None):
    """Yield (header, sequence, quality_string) from a FASTQ source."""
    lines = []
    if isinstance(source, str) and '\n' in source:
        fh = io.StringIO(source)
    else:
        opener = gzip.open if str(source).endswith('.gz') else open
        fh = opener(source, 'rt')
    try:
        count = 0
        it = iter(fh)
        for line in it:
            header = line.rstrip('\n')[1:]   # drop '@'
            seq    = next(it).rstrip('\n')
            next(it)                          # skip '+'
            qual   = next(it).rstrip('\n')
            yield header, seq, qual
            count += 1
            if max_reads and count >= max_reads:
                break
    finally:
        if not isinstance(source, str):
            fh.close()

def phred(char, offset=33):
    return ord(char) - offset

def mean_quality(qual_str):
    scores = [phred(c) for c in qual_str]
    return sum(scores) / len(scores)

# ── Sample FASTQ ──────────────────────────────────────────────────────────
sample_fastq = """\
@read1 length=50
ACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTACGTAC
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIA
@read2 length=50
TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT
+
!"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQR
"""

for hdr, seq, qual in parse_fastq(sample_fastq):
    q_vals = [phred(c) for c in qual]
    print(f"Read : {hdr}")
    print(f"  Seq len : {len(seq)}")
    print(f"  Mean Q  : {mean_quality(qual):.1f}")
    print(f"  Min Q   : {min(q_vals)}  Max Q: {max(q_vals)}")
    print(f"  Q≥30 %  : {sum(q>=30 for q in q_vals)/len(q_vals)*100:.1f}%")
    print()


In [ ]:
# ── FASTQ quality histogram (text-based) ─────────────────────────────────
from collections import Counter

qual_string = 'IIIIIIIIIII888888888!!!!!!!!!!!!!!!IIIIIIIIIIIIIIII'
counts = Counter(phred(c) for c in qual_string)

print(f"Quality score distribution (n={len(qual_string)} bases):")
print(f"{'Q':>4}  {'count':>5}  bar")
for q in sorted(counts):
    bar = '█' * counts[q]
    print(f"{q:>4}  {counts[q]:>5}  {bar}")


---

## 3. SAM / BAM / CRAM

### 3.1 SAM format

**SAM (Sequence Alignment/Map)** is the universal format for read alignments.

```
@HD  VN:1.6  SO:coordinate          ← header section (lines starting with @)
@SQ  SN:chr1  LN:248956422
@PG  ID:bwa  PN:bwa  VN:0.7.17
read1  0  chr1  100  60  50M  *  0  0  ACGT...  IIII...  NM:i:0
```

#### SAM FLAG field (column 2)

The FLAG is a bitwise combination:

| Bit  | Hex  | Meaning |
|------|------|---------|
| 1    | 0x1  | Read is paired |
| 2    | 0x2  | Pair properly mapped |
| 4    | 0x4  | Read unmapped |
| 8    | 0x8  | Mate unmapped |
| 16   | 0x10 | Read on reverse strand |
| 32   | 0x20 | Mate on reverse strand |
| 64   | 0x40 | Read 1 of pair |
| 128  | 0x80 | Read 2 of pair |
| 256  | 0x100 | Secondary alignment |
| 512  | 0x200 | Not passing filters |
| 1024 | 0x400 | PCR / optical duplicate |
| 2048 | 0x800 | Supplementary alignment |

#### CIGAR string

Describes how the read aligns to the reference:

| Op | Meaning |
|----|---------|
| M  | Match / mismatch (consumes both) |
| I  | Insertion to reference |
| D  | Deletion from reference |
| N  | Skipped region (intron in RNA-seq) |
| S  | Soft clip (bases present in read) |
| H  | Hard clip (bases removed) |
| =  | Sequence match |
| X  | Sequence mismatch |

### 3.2 BAM – Binary SAM

BAM is the **BGZF-compressed binary encoding** of SAM. It supports O(1) random
access via a `.bai` index created by `samtools index`.

### 3.3 CRAM – Compact Random Access Map

CRAM achieves ~50–60 % compression over BAM by storing differences from a
reference genome rather than raw bases. Requires access to the reference
at decoding time.

| Format | Typical size | Random access | Tools |
|--------|-------------|--------------|-------|
| SAM    | ~5–10 GB    | Sequential   | text editors, awk |
| BAM    | ~1–3 GB     | ✓ (with .bai) | samtools, pysam |
| CRAM   | ~0.5–1.5 GB | ✓ (with .crai) | samtools, htslib |


In [ ]:
# ── SAM FLAG decoder ─────────────────────────────────────────────────────
FLAG_BITS = {
    1:    'paired',
    2:    'proper_pair',
    4:    'unmapped',
    8:    'mate_unmapped',
    16:   'reverse_strand',
    32:   'mate_reverse_strand',
    64:   'read1',
    128:  'read2',
    256:  'secondary',
    512:  'qc_fail',
    1024: 'duplicate',
    2048: 'supplementary',
}

def decode_flag(flag):
    """Return a dict of set flag bits."""
    return {name: bool(flag & bit) for bit, name in FLAG_BITS.items()}

def flag_summary(flag):
    bits_set = [name for bit, name in FLAG_BITS.items() if flag & bit]
    return ', '.join(bits_set) if bits_set else 'none'

# Typical Illumina paired-end properly-mapped read1
for flag in [99, 147, 4, 1024, 2064]:
    print(f"FLAG {flag:>5} (0x{flag:04X}): {flag_summary(flag)}")


In [ ]:
# ── CIGAR string parser ───────────────────────────────────────────────────
import re as _re

CIGAR_OPS = {
    'M': ('consumes_query', 'consumes_reference'),
    'I': ('consumes_query',),
    'D': ('consumes_reference',),
    'N': ('consumes_reference',),
    'S': ('consumes_query',),
    'H': (),
    'P': (),
    '=': ('consumes_query', 'consumes_reference'),
    'X': ('consumes_query', 'consumes_reference'),
}

def parse_cigar(cigar_str):
    """Yield (length, op) tuples from a CIGAR string."""
    return [(int(n), op) for n, op in _re.findall(r'(\d+)([MIDNSHP=X])', cigar_str)]

def cigar_aligned_length(cigar_str):
    """Reference bases consumed by a CIGAR string (i.e., the alignment span)."""
    total = 0
    for length, op in parse_cigar(cigar_str):
        if 'consumes_reference' in CIGAR_OPS.get(op, ()):
            total += length
    return total

def cigar_read_length(cigar_str):
    """Query bases consumed (original read length)."""
    total = 0
    for length, op in parse_cigar(cigar_str):
        if 'consumes_query' in CIGAR_OPS.get(op, ()):
            total += length
    return total

# Examples
for cigar in ['50M', '25M5I20M', '10S35M5N10M', '3S45M2D3M']:
    ops = parse_cigar(cigar)
    print(f"{cigar:<20}  ref_span={cigar_aligned_length(cigar):>4}  read_len={cigar_read_length(cigar):>4}  ops={ops}")


---

## 4. VCF / BCF

### 4.1 VCF format

**VCF (Variant Call Format)** stores SNPs, indels, CNVs and structural variants.

```
##fileformat=VCFv4.3
##FILTER=<ID=PASS,Description="All filters passed">
##INFO=<ID=DP,Number=1,Type=Integer,Description="Total Depth">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
#CHROM  POS  ID  REF  ALT  QUAL  FILTER  INFO               FORMAT  SAMPLE1
chr1    925952  rs1234  G   A    50   PASS  DP=35;AF=0.5  GT:GQ  0/1:99
```

#### Key columns

| Col | Name | Description |
|-----|------|-------------|
| 1 | CHROM | Chromosome |
| 2 | POS | 1-based position |
| 3 | ID | dbSNP ID or `.` |
| 4 | REF | Reference allele |
| 5 | ALT | Alternate allele(s), comma-separated |
| 6 | QUAL | Phred-scaled call quality |
| 7 | FILTER | PASS or filter names |
| 8 | INFO | Semicolon-separated key=value annotations |
| 9 | FORMAT | Colon-separated keys for sample columns |
| 10+ | SAMPLES | Genotype data for each sample |

### 4.2 Genotype field

```
GT  0/0  = homozygous reference
GT  0/1  = heterozygous
GT  1/1  = homozygous alternate
GT  0|1  = phased heterozygous (| instead of /)
GT  ./.  = missing genotype
```

### 4.3 BCF

BCF is the binary equivalent of VCF (like BAM is to SAM). Use
`bcftools view` to convert between them.


In [ ]:
# ── Lightweight VCF parser ────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Dict, List, Optional

@dataclass
class VCFRecord:
    chrom: str
    pos: int            # 1-based
    id: str
    ref: str
    alt: List[str]
    qual: Optional[float]
    filter: List[str]
    info: Dict[str, str]
    format_keys: List[str] = field(default_factory=list)
    samples: List[Dict[str, str]] = field(default_factory=list)

    @property
    def is_snp(self):
        return all(len(a) == 1 and len(self.ref) == 1 for a in self.alt)

    @property
    def is_indel(self):
        return any(len(a) != len(self.ref) for a in self.alt)

def parse_vcf(text):
    """Yield VCFRecord objects; skip meta-information lines."""
    header_samples = []
    for line in text.splitlines():
        if line.startswith('##'):
            continue
        if line.startswith('#CHROM'):
            header_samples = line.lstrip('#').split('\t')[9:]
            continue
        parts = line.split('\t')
        if len(parts) < 8:
            continue
        chrom, pos, vid, ref, alt_str, qual_str, filt_str, info_str = parts[:8]
        alt = alt_str.split(',')
        try:
            qual = float(qual_str)
        except ValueError:
            qual = None
        filt = filt_str.split(';') if filt_str != '.' else []
        info = {}
        for item in info_str.split(';'):
            if '=' in item:
                k, v = item.split('=', 1)
                info[k] = v
            else:
                info[item] = True
        fmt_keys = parts[8].split(':') if len(parts) > 8 else []
        samples = []
        for s_str in parts[9:]:
            s_vals = s_str.split(':')
            samples.append(dict(zip(fmt_keys, s_vals)))
        yield VCFRecord(chrom, int(pos), vid, ref, alt, qual,
                        filt, info, fmt_keys, samples)

# ── Sample VCF ────────────────────────────────────────────────────────────
sample_vcf = """\
##fileformat=VCFv4.3
##INFO=<ID=DP,Number=1,Type=Integer,Description="Read Depth">
##INFO=<ID=AF,Number=A,Type=Float,Description="Allele Frequency">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSAMPLE
chr1\t925952\trs3115860\tG\tA\t50\tPASS\tDP=35;AF=0.5\tGT:GQ\t0/1:99
chr1\t1234567\t.\tATT\tA\t30\tPASS\tDP=20;AF=0.3\tGT:GQ\t0/1:75
chr2\t8888888\trs9999\tC\tT,G\t99\tPASS\tDP=100;AF=0.45,0.05\tGT:GQ\t1/2:80
"""

for rec in parse_vcf(sample_vcf):
    variant_type = 'SNP' if rec.is_snp else 'INDEL' if rec.is_indel else 'COMPLEX'
    gt = rec.samples[0].get('GT', './.') if rec.samples else '?'
    print(f"{rec.chrom}:{rec.pos}  {rec.ref}→{','.join(rec.alt)}  "          f"{variant_type:<8}  DP={rec.info.get('DP','?'):<4}  GT={gt}")


---

## 5. BED – Genomic Intervals

### 5.1 Format specification

BED (Browser Extensible Data) stores genomic intervals in a tab-separated text
file. The coordinate system is **0-based, half-open** [start, end).

```
chr1    0       1000    region_A    100    +
chr1    5000    6000    region_B    200    -
```

#### Mandatory and optional columns

| Col | Name | Required | Description |
|-----|------|----------|-------------|
| 1 | chrom | ✓ | Chromosome |
| 2 | chromStart | ✓ | 0-based start |
| 3 | chromEnd | ✓ | End (not included) |
| 4 | name | optional | Feature name |
| 5 | score | optional | 0–1000 |
| 6 | strand | optional | `+` or `-` |
| 7 | thickStart | optional | Coding start (CDS) |
| 8 | thickEnd | optional | Coding end |
| 9 | itemRgb | optional | `R,G,B` colour |
| 10–12 | block* | optional | Exon block info |

### 5.2 BED vs 1-based formats

```
Sequence:  A  T  G  C  T  A  G
Index:     0  1  2  3  4  5  6  (BED / Python)
           1  2  3  4  5  6  7  (SAM POS / VCF)

BED: start=1, end=4  →  T G C  (length 3)
SAM/VCF: POS=2       →  same T position
```


In [ ]:
# ── BED interval toolkit ──────────────────────────────────────────────────
from dataclasses import dataclass
from typing import Optional

@dataclass
class BEDRecord:
    chrom: str
    start: int   # 0-based
    end: int     # exclusive
    name: Optional[str] = None
    score: Optional[int] = None
    strand: Optional[str] = None

    @property
    def length(self):
        return self.end - self.start

    @property
    def center(self):
        return (self.start + self.end) // 2

    def overlaps(self, other):
        return (self.chrom == other.chrom and
                self.start < other.end and
                other.start < self.end)

    def distance(self, other):
        if self.chrom != other.chrom:
            return float('inf')
        if self.overlaps(other):
            return 0
        return max(self.start, other.start) - min(self.end, other.end)

def parse_bed(text):
    """Yield BEDRecord objects from tab-separated text (skips # and track lines)."""
    for line in text.splitlines():
        if not line or line.startswith('#') or line.startswith('track'):
            continue
        parts = line.split('\t')
        chrom, start, end = parts[0], int(parts[1]), int(parts[2])
        name   = parts[3] if len(parts) > 3 else None
        score  = int(parts[4]) if len(parts) > 4 and parts[4] != '.' else None
        strand = parts[5] if len(parts) > 5 else None
        yield BEDRecord(chrom, start, end, name, score, strand)

# ── Example ───────────────────────────────────────────────────────────────
sample_bed = """\
chr1\t1000\t2000\tgene_A\t500\t+
chr1\t1500\t2500\tgene_B\t300\t-
chr1\t5000\t6000\tgene_C\t700\t+
chr2\t100\t200\tgene_D\t100\t.
"""

records = list(parse_bed(sample_bed))
for r in records:
    print(f"{r.chrom}:{r.start}-{r.end}  name={r.name:<8}  len={r.length}  strand={r.strand}")

print()
# Check overlaps
A, B, C = records[0], records[1], records[2]
print(f"A overlaps B? {A.overlaps(B)}")
print(f"A overlaps C? {A.overlaps(C)}")
print(f"Distance A→C: {A.distance(C)} bp")


---

## 6. GFF / GTF – Gene Annotations

### 6.1 GFF3 format

**GFF3 (General Feature Format 3)** is the standard for genome annotation.

```
##gff-version 3
chr1  Ensembl  gene   11869  14409  .  +  .  ID=ENSG00000223972;Name=DDX11L1
chr1  Ensembl  mRNA   11869  14409  .  +  .  ID=ENST00000456328;Parent=ENSG00000223972
chr1  Ensembl  exon   11869  12227  .  +  .  Parent=ENST00000456328
```

Columns (1-based coordinates, like VCF):

| Col | Name | Description |
|-----|------|-------------|
| 1 | seqname | Chromosome or scaffold |
| 2 | source | Program/database that produced the feature |
| 3 | feature | gene, mRNA, exon, CDS, UTR, … |
| 4 | start | 1-based start |
| 5 | end | 1-based, inclusive end |
| 6 | score | Numeric or `.` |
| 7 | strand | `+`, `-`, `.`, or `?` |
| 8 | phase | 0, 1, 2 for CDS; `.` otherwise |
| 9 | attributes | Semicolon-separated key=value pairs |

### 6.2 GTF (Ensembl/GENCODE)

GTF (Gene Transfer Format) is functionally similar to GFF2 and used by
Ensembl/GENCODE and RNA-seq tools like HISAT2, StringTie:

```
chr1  HAVANA  gene  11869  14409  .  +  .  gene_id "ENSG00000223972"; gene_name "DDX11L1";
```

Key differences from GFF3:

| Feature | GFF3 | GTF |
|---------|------|-----|
| Attribute separator | `;` (key=value) | `; ` (key "value") |
| Hierarchy | Explicit `Parent=` | Implicit via `gene_id`/`transcript_id` |
| Phase column | 8 (0/1/2/.) | 8 (0/1/2/.) |


In [ ]:
# ── GFF3 parser ───────────────────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import Dict, Any, Optional, List

@dataclass
class GFFRecord:
    seqname: str
    source: str
    feature: str
    start: int   # 1-based
    end: int     # inclusive
    score: Optional[float]
    strand: str
    phase: Optional[int]
    attributes: Dict[str, Any] = field(default_factory=dict)

    @property
    def length(self):
        return self.end - self.start + 1

def _parse_gff3_attrs(attr_str):
    attrs = {}
    for item in attr_str.strip().rstrip(';').split(';'):
        item = item.strip()
        if not item:
            continue
        if '=' in item:
            k, v = item.split('=', 1)
            attrs[k.strip()] = v.strip()
    return attrs

def parse_gff3(text):
    """Yield GFFRecord objects from GFF3 text."""
    for line in text.splitlines():
        if not line or line.startswith('#'):
            continue
        parts = line.split('\t')
        if len(parts) < 8:
            continue
        seqname, source, feature = parts[0], parts[1], parts[2]
        start, end = int(parts[3]), int(parts[4])
        score = None if parts[5] == '.' else float(parts[5])
        strand = parts[6]
        phase = None if parts[7] == '.' else int(parts[7])
        attrs = _parse_gff3_attrs(parts[8]) if len(parts) > 8 else {}
        yield GFFRecord(seqname, source, feature, start, end,
                        score, strand, phase, attrs)

# ── Sample annotation ─────────────────────────────────────────────────────
sample_gff3 = """\
##gff-version 3
chr1\tEnsembl\tgene\t11869\t14409\t.\t+\t.\tID=ENSG00000223972;Name=DDX11L1;biotype=lncRNA
chr1\tEnsembl\tmRNA\t11869\t14409\t.\t+\t.\tID=ENST00000456328;Parent=ENSG00000223972
chr1\tEnsembl\texon\t11869\t12227\t.\t+\t.\tParent=ENST00000456328;exon_number=1
chr1\tEnsembl\texon\t12613\t12721\t.\t+\t.\tParent=ENST00000456328;exon_number=2
chr1\tEnsembl\texon\t13221\t14409\t.\t+\t.\tParent=ENST00000456328;exon_number=3
"""

for rec in parse_gff3(sample_gff3):
    name = rec.attributes.get('Name', rec.attributes.get('ID', '?'))
    print(f"{rec.feature:<6}  {rec.seqname}:{rec.start}-{rec.end}  "          f"len={rec.length:<6}  strand={rec.strand}  name={name}")


---

## 7. WIG / BedGraph / BigWig – Coverage Tracks

These formats store per-base or per-interval numerical values (coverage depth,
ChIP-seq signal, conservation scores, etc.).

### 7.1 BedGraph

The simplest coverage format – four tab-separated columns.
**0-based, half-open** coordinates (like BED).

```
track type=bedGraph name="Coverage"
chr1    0     100   5
chr1    100   200   12
chr1    200   300   0
```

### 7.2 WIG (fixed-step and variable-step)

```
# Fixed-step: evenly spaced positions
fixedStep chrom=chr1 start=1 step=1 span=1
5
12
8

# Variable-step: positions with gaps
variableStep chrom=chr1 span=1
100    5
250    12
501    8
```

### 7.3 BigWig / BigBed

Binary, indexed versions of WIG and BED respectively.
- Created with `wigToBigWig` / `bedToBigBed` (UCSC tools) or `pyBigWig`
- Random-access via index; efficient for genome browsers and large-scale analyses
- Cannot be read with standard text tools; use `pyBigWig` or `bigWigToBedGraph`

| Format | Text/Binary | Random access | Memory | Typical use |
|--------|-------------|--------------|--------|-------------|
| WIG | Text | No | High | Small tracks |
| BedGraph | Text | No | High | Differential signal |
| BigWig | Binary | ✓ | Low | Genome browsers, deepTools |
| BigBed | Binary | ✓ | Low | Large BED feature sets |


In [ ]:
# ── BedGraph parser ───────────────────────────────────────────────────────
from dataclasses import dataclass
from typing import List, Tuple

@dataclass
class BedGraphRecord:
    chrom: str
    start: int   # 0-based
    end: int
    value: float

def parse_bedgraph(text):
    """Yield BedGraphRecord objects (skip track/browser lines)."""
    for line in text.splitlines():
        if not line or line.startswith('track') or line.startswith('browser'):
            continue
        parts = line.split('\t')
        yield BedGraphRecord(parts[0], int(parts[1]), int(parts[2]), float(parts[3]))

def mean_coverage(records: List[BedGraphRecord]) -> float:
    """Weighted mean coverage over all intervals."""
    total_bases = total_cov = 0
    for r in records:
        length = r.end - r.start
        total_bases += length
        total_cov   += r.value * length
    return total_cov / total_bases if total_bases else 0.0

# ── Example ───────────────────────────────────────────────────────────────
sample_bedgraph = """\
track type=bedGraph name=RNAseq_coverage
chr1\t0\t100\t5.2
chr1\t100\t250\t12.8
chr1\t250\t400\t0.0
chr1\t400\t500\t7.5
"""

records = list(parse_bedgraph(sample_bedgraph))
for r in records:
    bar = '▓' * int(r.value)
    print(f"{r.chrom}:{r.start:>6}-{r.end:<6}  {r.value:>6.1f}  {bar}")
print(f"\nMean coverage: {mean_coverage(records):.2f}x")


---

## 8. PDB – Protein Data Bank Format

### 8.1 Overview

The legacy PDB flat-file format stores 3-D atomic coordinates of macromolecules.
Despite being superseded by mmCIF for deposition, millions of structures exist
only in PDB format and many tools still accept it.

### 8.2 Key record types

```
HEADER    OXIDOREDUCTASE                          23-FEB-94   1HHO
TITLE     HUMAN OXYHAEMOGLOBIN AT 2.1 ANGSTROMS
SEQRES    1 A  141  VAL LEU SER PRO ALA ...
ATOM      1  N   ALA A   1      11.104  13.207  21.106  1.00 12.06     N
HETATM  800  C1  HEM A 141      30.000  31.000  22.000  1.00 15.00     C
CONECT  800  801  802
END
```

#### ATOM / HETATM columns (fixed-width!)

```
Columns  1-6:  Record type (ATOM or HETATM)
        7-11:  Serial number
       13-16:  Atom name (space-padded)
          17:  Alternate location indicator
       18-20:  Residue name
          22:  Chain ID
       23-26:  Residue sequence number
          27:  Code for insertions
       31-38:  X (Å)
       39-46:  Y (Å)
       47-54:  Z (Å)
       55-60:  Occupancy
       61-66:  Temperature factor (B-factor)
       77-78:  Element symbol
       79-80:  Charge
```

> ⚠️ PDB format uses **fixed-column widths** — parsing requires exact slicing,
> not `split()`.


In [ ]:
# ── Minimal PDB ATOM/HETATM parser ──────────────────────────────────────
from dataclasses import dataclass
from typing import Optional, List, Dict

@dataclass
class PDBAtom:
    record: str        # ATOM or HETATM
    serial: int
    name: str          # atom name
    alt_loc: str
    res_name: str
    chain: str
    res_seq: int
    icode: str         # insertion code
    x: float
    y: float
    z: float
    occupancy: float
    b_factor: float
    element: str

def parse_pdb_atoms(text: str) -> List[PDBAtom]:
    """Parse ATOM and HETATM records from PDB text."""
    atoms = []
    for line in text.splitlines():
        if not (line.startswith('ATOM') or line.startswith('HETATM')):
            continue
        line = line.ljust(80)   # pad to 80 cols if shorter
        atoms.append(PDBAtom(
            record   = line[0:6].strip(),
            serial   = int(line[6:11]),
            name     = line[12:16].strip(),
            alt_loc  = line[16].strip(),
            res_name = line[17:20].strip(),
            chain    = line[21].strip(),
            res_seq  = int(line[22:26]),
            icode    = line[26].strip(),
            x        = float(line[30:38]),
            y        = float(line[38:46]),
            z        = float(line[46:54]),
            occupancy= float(line[54:60]),
            b_factor = float(line[60:66]),
            element  = line[76:78].strip() if len(line) > 76 else '',
        ))
    return atoms

def backbone_atoms(atoms: List[PDBAtom]) -> List[PDBAtom]:
    """Return only backbone N, CA, C, O atoms."""
    return [a for a in atoms if a.record == 'ATOM'
            and a.name in ('N', 'CA', 'C', 'O')]

# ── Sample PDB fragment (1HHO α-chain first residue) ─────────────────────
sample_pdb = """\
ATOM      1  N   VAL A   1      -4.456   3.545  24.646  1.00 17.50           N
ATOM      2  CA  VAL A   1      -3.188   3.945  23.990  1.00 15.73           C
ATOM      3  C   VAL A   1      -2.040   3.076  24.476  1.00 14.58           C
ATOM      4  O   VAL A   1      -1.989   1.919  24.080  1.00 15.73           O
ATOM      5  CB  VAL A   1      -3.267   3.867  22.467  1.00 17.15           C
ATOM      6  N   LEU A   2      -1.119   3.617  25.288  1.00 13.87           N
ATOM      7  CA  LEU A   2       0.048   2.868  25.749  1.00 12.64           C
HETATM  800  FE  HEM A 141      25.000  26.000  20.000  1.00 14.00          FE
"""

atoms = parse_pdb_atoms(sample_pdb)
print(f"Total atoms parsed: {len(atoms)}")
print(f"ATOM records: {sum(1 for a in atoms if a.record=='ATOM')}")
print(f"HETATM records: {sum(1 for a in atoms if a.record=='HETATM')}")
print()
for a in atoms[:5]:
    print(f"  {a.record:<6} {a.serial:>4}  {a.name:<4} {a.res_name} {a.chain}{a.res_seq}  "          f"({a.x:7.3f}, {a.y:7.3f}, {a.z:7.3f})  B={a.b_factor}")


In [ ]:
# ── Distance calculation ─────────────────────────────────────────────────
import math

def distance(a1: PDBAtom, a2: PDBAtom) -> float:
    """Euclidean distance in Angstroms between two atoms."""
    return math.sqrt((a1.x-a2.x)**2 + (a1.y-a2.y)**2 + (a1.z-a2.z)**2)

def find_contacts(atoms: List[PDBAtom], cutoff: float = 3.5) -> List[tuple]:
    """Return atom pairs within cutoff Angstroms (brute-force O(n²))."""
    contacts = []
    for i, a1 in enumerate(atoms):
        for a2 in atoms[i+1:]:
            if distance(a1, a2) <= cutoff:
                contacts.append((a1, a2, distance(a1, a2)))
    return sorted(contacts, key=lambda x: x[2])

# Demonstrate with our sample backbone atoms
bb = backbone_atoms(atoms)
print(f"Backbone atoms: {len(bb)}")
contacts = find_contacts(bb, cutoff=4.0)
print(f"Backbone atom pairs within 4.0 Å:")
for a1, a2, d in contacts:
    print(f"  {a1.name}({a1.res_name}{a1.res_seq}) -- {a2.name}({a2.res_name}{a2.res_seq}): {d:.2f} Å")


---

## 9. mmCIF / PDBx – Modern Structure Format

### 9.1 Why mmCIF?

The legacy PDB format has a hard limit of:
- 99 999 atoms (5-digit serial)
- 9 999 residues per chain
- Single-letter chain IDs (26 chains)

**mmCIF (macromolecular Crystallographic Information File)** removes all these
limits and is now the **mandatory submission format** for the PDB.

### 9.2 Format structure

mmCIF is a hierarchical key-value / loop format:

```cif
data_1ABC
#
_entry.id   1ABC
_struct.title  'Human haemoglobin'
#
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
ATOM 1 N N  VAL A 1  -4.456  3.545 24.646 1.00 17.50
ATOM 2 C CA VAL A 1  -3.188  3.945 23.990 1.00 15.73
```

### 9.3 Parsing mmCIF with BioPython / gemmi

```python
# BioPython (MMCIFParser)
from Bio.PDB import MMCIFParser
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("1ABC", "1abc.cif")

# gemmi (preferred for large structures)
import gemmi
structure = gemmi.read_structure("1abc.cif")
```

### 9.4 AlphaFold & ModelCIF

AlphaFold2/3 predicted structures are distributed as mmCIF files with
additional ModelCIF metadata including pLDDT confidence scores stored in the
B-factor column.

```
B-factor column in AlphaFold CIF = pLDDT (0–100)
  > 90  = very high confidence (dark blue)
  70–90 = confident (light blue)
  50–70 = low confidence (yellow)
  < 50  = very low (orange)
```


In [ ]:
# ── Minimal mmCIF atom_site loop reader ─────────────────────────────────
import re

def parse_mmcif_atom_site(text: str):
    """
    Very lightweight parser for the _atom_site loop in a mmCIF file.
    Yields dicts keyed by the column names (without '_atom_site.' prefix).
    """
    # Find the atom_site loop block
    match = _re.search(
        r'loop_\s+((?:_atom_site\.\S+\s*)+)(.*?)(?=loop_|#|\Z)',
        text, _re.DOTALL
    )
    if not match:
        return
    header_block, data_block = match.group(1), match.group(2)
    # Extract column names
    columns = [c.split('.')[1].strip()
               for c in _re.findall(r'_atom_site\.\S+', header_block)]
    # Tokenize data block
    tokens = _re.findall(r"'[^']*'|\S+", data_block)
    n = len(columns)
    for i in range(0, len(tokens) - n + 1, n):
        row = tokens[i:i+n]
        if len(row) == n and row[0] in ('ATOM', 'HETATM'):
            yield dict(zip(columns, row))

# ── Tiny mmCIF sample ─────────────────────────────────────────────────────
sample_mmcif = """
data_DEMO
loop_
_atom_site.group_PDB
_atom_site.id
_atom_site.type_symbol
_atom_site.label_atom_id
_atom_site.label_comp_id
_atom_site.label_asym_id
_atom_site.label_seq_id
_atom_site.Cartn_x
_atom_site.Cartn_y
_atom_site.Cartn_z
_atom_site.occupancy
_atom_site.B_iso_or_equiv
ATOM 1 N N  VAL A 1 -4.456  3.545 24.646 1.00 17.50
ATOM 2 C CA VAL A 1 -3.188  3.945 23.990 1.00 15.73
ATOM 3 C C  VAL A 1 -2.040  3.076 24.476 1.00 14.58
#
"""

rows = list(parse_mmcif_atom_site(sample_mmcif))
print(f"Parsed {len(rows)} atom_site rows")
if rows:
    print(f"Columns: {list(rows[0].keys())}")
    print()
    for r in rows:
        print(f"  {r['group_PDB']} {r['id']:>3}  {r['label_atom_id']:<4} "              f"{r['label_comp_id']} chain={r['label_asym_id']}  "              f"({float(r['Cartn_x']):7.3f}, {float(r['Cartn_y']):7.3f}, {float(r['Cartn_z']):7.3f})")


---

## 10. FAST5 / POD5 – Oxford Nanopore Raw Signal

### 10.1 FAST5

FAST5 is an HDF5 container used by early Oxford Nanopore devices to store:
- Raw ionic-current signal (integer array, picoamperes × scale)
- Basecalled sequences and quality scores
- Run metadata (flow cell, chemistry, software versions)

```python
import h5py
with h5py.File('read.fast5', 'r') as f:
    # Navigate the group hierarchy
    reads = list(f.keys())          # e.g. ['read_abc123']
    raw = f[reads[0]]['Raw']['Signal'][()]   # numpy array of raw signal
    events = f[reads[0]]['Analyses']['Basecall_1D_000']
    fastq = events['BaseCalled_template']['Fastq'][()].decode()
```

### 10.2 POD5 (successor to FAST5)

POD5 is Oxford Nanopore's next-generation format (2022+):
- Based on **Apache Arrow** columnar storage
- ~3× smaller than FAST5 at equivalent compression
- Supports batch reading for high throughput

```python
import pod5
with pod5.Reader("reads.pod5") as reader:
    for read in reader.reads():
        signal = read.signal          # numpy array
        sequence = read.sequence      # basecalled string (if present)
        quality  = read.quality       # quality string
```

### 10.3 Basecalling and format conversion

```bash
# Convert multi-FAST5 → POD5
pod5 convert fast5 *.fast5 --output reads.pod5

# Basecall with Dorado (ONT's current basecaller)
dorado basecaller dna_r10.4.1_e8.2_400bps_hac@v4.3.0 reads.pod5 > reads.bam

# Extract FASTQ from BAM
samtools fastq reads.bam > reads.fastq
```

| Format | Size | Read access | Metadata | Status |
|--------|------|-------------|----------|--------|
| FAST5 (HDF5) | Large | Single-read | Rich | Legacy |
| Multi-FAST5 | Medium | Batch | Rich | Common |
| POD5 | Small | Batch | Rich | Current standard |


In [ ]:
# ── Simulating FAST5-style raw signal ────────────────────────────────────
import math, random

def simulate_nanopore_signal(sequence: str, mean_level: float = 100.0,
                              noise_sd: float = 5.0,
                              samples_per_base: int = 10) -> list:
    """
    Very simplified model: each base generates a block of current samples.
    Real signals are far more complex (dwell-time variation, context effects).
    """
    # Rough level offsets per base (not biologically accurate)
    level_offset = {'A': 0, 'C': 10, 'G': -10, 'T': 5}
    signal = []
    for base in sequence.upper():
        level = mean_level + level_offset.get(base, 0)
        for _ in range(samples_per_base):
            signal.append(level + random.gauss(0, noise_sd))
    return signal

random.seed(42)
seq = 'ACGTACGTACGT'
signal = simulate_nanopore_signal(seq)

print(f"Sequence: {seq}  ({len(seq)} bases)")
print(f"Signal samples: {len(signal)}  ({len(signal)/len(seq):.0f} samples/base)")
print(f"Signal range: [{min(signal):.1f}, {max(signal):.1f}]")
print(f"Mean: {sum(signal)/len(signal):.1f}  pA (simulated)")

# ASCII visualisation
buckets = 20
mn, mx = min(signal), max(signal)
hist = [0] * buckets
for v in signal:
    idx = min(int((v - mn) / (mx - mn) * buckets), buckets - 1)
    hist[idx] += 1
print("\nSignal histogram:")
for i, h in enumerate(hist):
    lo = mn + (mx-mn)*i/buckets
    print(f"  {lo:6.1f}  {'█'*h}")


---

## 11. Newick / Nexus / NHX – Phylogenetic Trees

### 11.1 Newick format

Newick is the simplest tree serialization: a nested parenthesis string.

```
((Human:0.01, Chimpanzee:0.015):0.05, (Mouse:0.09, Rat:0.08):0.04):0.0;
  └── internal node ───────────────┘   └─ internal node ──────────┘
```

Rules:
- Leaf names are unquoted strings (or `'quoted strings'`).
- `:branch_length` follows each node (optional).
- Each subtree is enclosed in `()`.
- The tree ends with `;`.

### 11.2 Nexus format

Nexus is a richer container used by MrBayes, PAUP*, and FigTree.
It can store multiple data blocks (sequences, trees, characters):

```nexus
#NEXUS
BEGIN TAXA;
  NTAX = 4;
  TAXLABELS Human Chimpanzee Mouse Rat;
END;
BEGIN TREES;
  TREE tree1 = ((Human:0.01,Chimpanzee:0.015):0.05,(Mouse:0.09,Rat:0.08):0.04);
END;
```

### 11.3 NHX – New Hampshire Extended

NHX adds key-value annotations to Newick nodes using `[&&NHX:key=value]`:

```
((Human:0.01[&&NHX:S=Homo_sapiens:D=N],Chimp:0.015[&&NHX:S=Pan_troglodytes:D=N])
[&&NHX:D=N:B=95]:0.05);
```

Common NHX tags:

| Tag | Meaning |
|-----|---------|
| S=  | Species name |
| D=  | Duplication (Y/N) |
| B=  | Bootstrap support |
| E=  | EC number |
| O=  | Sequence identifier |

### 11.4 Other formats

| Format | Extension | Notes |
|--------|-----------|-------|
| PhyloXML | `.xml` | XML tree with rich metadata |
| NeXML | `.xml` | W3C-compliant XML tree format |
| BEAST | `.xml` | Input/output for BEAST Bayesian analysis |
| iTOL annotation | `.txt` | Annotation files for Interactive Tree of Life |


In [ ]:
# ── Recursive Newick parser ───────────────────────────────────────────────
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class TreeNode:
    name: str = ''
    branch_length: Optional[float] = None
    children: List['TreeNode'] = field(default_factory=list)

    @property
    def is_leaf(self):
        return len(self.children) == 0

    def num_leaves(self):
        if self.is_leaf:
            return 1
        return sum(c.num_leaves() for c in self.children)

    def all_leaves(self):
        if self.is_leaf:
            yield self
        for c in self.children:
            yield from c.all_leaves()

    def depth(self):
        if self.is_leaf:
            return 0
        return 1 + max(c.depth() for c in self.children)

def parse_newick(s: str) -> TreeNode:
    """Parse a Newick string and return the root TreeNode."""
    s = s.strip().rstrip(';')

    def _parse(pos):
        node = TreeNode()
        if pos < len(s) and s[pos] == '(':
            pos += 1  # consume '('
            while True:
                child, pos = _parse(pos)
                node.children.append(child)
                if pos >= len(s) or s[pos] == ')':
                    pos += 1  # consume ')'
                    break
                if s[pos] == ',':
                    pos += 1  # consume ','
        # Node label (name + optional :branch_length)
        name_chars, bl_chars = [], []
        reading_bl = False
        while pos < len(s) and s[pos] not in (',', ')', ';'):
            ch = s[pos]
            if ch == ':':
                reading_bl = True
            elif reading_bl:
                bl_chars.append(ch)
            else:
                name_chars.append(ch)
            pos += 1
        node.name = ''.join(name_chars).strip()
        if bl_chars:
            node.branch_length = float(''.join(bl_chars))
        return node, pos

    root, _ = _parse(0)
    return root

def print_tree(node, indent=0, branch=''):
    connector = '├── ' if indent > 0 else ''
    bl = f" [{node.branch_length:.4f}]" if node.branch_length is not None else ''
    label = node.name if node.name else '(internal)'
    print(' ' * indent + connector + label + bl)
    for child in node.children:
        print_tree(child, indent + 4)

# ── Example trees ─────────────────────────────────────────────────────────
newick1 = """((Human:0.010,Chimpanzee:0.015):0.050,(Mouse:0.090,Rat:0.080):0.040);"""
newick2 = """(((A:1,B:2):3,(C:4,D:5):6):7,E:8);"""

print("=== Primate tree ===")
tree = parse_newick(newick1)
print_tree(tree)
print(f"\nLeaves: {[n.name for n in tree.all_leaves()]}")
print(f"Depth:  {tree.depth()}")
print(f"Tips:   {tree.num_leaves()}")


In [ ]:
# ── BioPython Phylo module ────────────────────────────────────────────────
if BIOPYTHON:
    from Bio import Phylo
    from io import StringIO

    newick_str = "((Human:0.010,Chimpanzee:0.015):0.050,(Mouse:0.090,Rat:0.080):0.040);"
    tree = Phylo.read(StringIO(newick_str), 'newick')

    print("Tree ASCII art (BioPython):")
    Phylo.draw_ascii(tree)

    print("\nAll clades:")
    for clade in tree.find_clades():
        print(f"  name={clade.name!r:<15}  branch_length={clade.branch_length}")

    # Total tree length (sum of all branch lengths)
    total = sum(c.branch_length for c in tree.find_clades()
                if c.branch_length is not None)
    print(f"\nTotal branch length: {total:.3f}")
else:
    print("BioPython not available – skipping Phylo demo.")


---

## 12. Quick-Reference Table

| Format | Ext | Coords | Use case | Python tool |
|--------|-----|--------|----------|-------------|
| FASTA | .fa .fna .faa | — | Sequences | `Bio.SeqIO`, built-in |
| FASTQ | .fq .fastq | — | Reads + quality | `Bio.SeqIO`, built-in |
| SAM | .sam | 1-based | Alignments (text) | `pysam`, built-in |
| BAM | .bam | 1-based | Alignments (binary) | `pysam` |
| CRAM | .cram | 1-based | Alignments (ref-compressed) | `pysam`, `htslib` |
| VCF | .vcf | 1-based | Variants (text) | `cyvcf2`, `pyvcf3` |
| BCF | .bcf | 1-based | Variants (binary) | `cyvcf2`, `bcftools` |
| BED | .bed | 0-based half-open | Intervals | `pybedtools`, built-in |
| GFF3 | .gff3 .gff | 1-based inclusive | Gene annotations | `gffutils`, built-in |
| GTF | .gtf | 1-based inclusive | Ensembl annotations | `pyranges`, built-in |
| WIG | .wig | 1-based | Coverage (text) | built-in |
| BedGraph | .bedgraph | 0-based | Coverage / signal | built-in |
| BigWig | .bw .bigwig | 0-based | Coverage (binary) | `pyBigWig` |
| BigBed | .bb .bigbed | 0-based | Features (binary) | `pyBigWig` |
| PDB | .pdb .ent | Å (3-D) | Structure (legacy) | `Bio.PDB`, `gemmi` |
| mmCIF | .cif .mmcif | Å (3-D) | Structure (current) | `Bio.PDB`, `gemmi` |
| FAST5 | .fast5 | — | Nanopore raw signal | `h5py` |
| POD5 | .pod5 | — | Nanopore raw signal | `pod5` |
| Newick | .nwk .tree | — | Phylogenetic trees | `Bio.Phylo`, `ete3` |
| Nexus | .nex .nexus | — | Multi-block phylo | `Bio.Phylo`, `ete3` |
| NHX | .nhx | — | Annotated trees | `ete3` |

---

### Recommended libraries

```bash
pip install biopython pysam cyvcf2 pybedtools gffutils pyBigWig gemmi pod5 ete3
```

For conda users:
```bash
conda install -c bioconda biopython pysam cyvcf2 pybedtools gffutils ucsc-bigwigtobedgraph
```


---

## Summary

You have now seen the full landscape of bioinformatics data formats:

| Category | Formats covered |
|----------|----------------|
| Sequence | FASTA, FASTQ |
| Alignment | SAM, BAM, CRAM |
| Variant | VCF, BCF |
| Interval / annotation | BED, GFF3, GTF |
| Signal / coverage | WIG, BedGraph, BigWig, BigBed |
| Structure | PDB, mmCIF/PDBx |
| Raw signal | FAST5, POD5 |
| Phylogeny | Newick, Nexus, NHX |

**Key take-aways**:
- Always check the **coordinate system** (0-based vs 1-based, half-open vs inclusive).
- Prefer **binary / indexed** formats (BAM, BCF, BigWig, POD5) for large-scale work.
- Use **BioPython** for standard parsing; reach for specialist libraries (`pysam`,
  `cyvcf2`, `pyBigWig`, `gemmi`) when performance or format coverage matters.
- **mmCIF** is the current PDB standard; PDB legacy format is still widely used.
- **POD5** replaces FAST5 for Oxford Nanopore going forward.


---

[< Previous: NGS Fundamentals: From Sequencing to Alignment](01_ngs_fundamentals.ipynb) | [Tier 3: Applied Bioinformatics Overview](../README.md) | [Next: Variant Calling and SNP Analysis >](../02_Variant_Calling_and_SNP_Analysis/01_variant_calling_and_snp_analysis.ipynb)

---